# Telco Churn 数据集 Governance 实验

本 notebook 用于运行第二篇 governance 框架在 Telco Churn 数据集上的实验。

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("当前项目目录：", PROJECT_ROOT)


当前项目目录： e:\yan\组\三支决策\机器学习\BT_TWD_v2


In [2]:
CONFIG_PATH = "configs/telco_churn.yaml"

# 是否运行消融实验：
# True：运行 no_cp 和 no_progressive 两个消融；False：只运行 full，跳过消融单元。
RUN_ABLATION = False

print("当前配置文件：", CONFIG_PATH)
print("运行消融实验：", RUN_ABLATION)


当前配置文件： configs/telco_churn.yaml
运行消融实验： False


## 结果摘要工具函数


In [3]:
import pandas as pd
from pathlib import Path
from IPython.display import display

output_root = Path("outputs/governance")

def show_governance_summary(mode: str) -> None:
    """Read and display the current dataset summary for one run mode."""
    config_stem = Path(CONFIG_PATH).stem
    print(f"\n===== {mode} results =====")

    dataset_summary = output_root / mode / "dataset_summary.csv"
    fold_summary = output_root / mode / config_stem / "fold_summary.csv"
    sample_records = output_root / mode / config_stem / "sample_records.csv"

    if dataset_summary.exists():
        print("Reading:", dataset_summary)
        summary_df = pd.read_csv(dataset_summary)
        if "dataset_name" in summary_df.columns:
            dataset_row = summary_df[summary_df["dataset_name"].astype(str) == config_stem]
            if dataset_row.empty:
                available = ", ".join(summary_df["dataset_name"].astype(str).tolist())
                print(f"No row for {config_stem}; available datasets: {available}")
            else:
                display(dataset_row.reset_index(drop=True))
        else:
            print("dataset_summary.csv has no dataset_name column; cannot filter by dataset.")
    else:
        print("Missing dataset_summary.csv:", dataset_summary)

    if fold_summary.exists():
        print("Reading:", fold_summary)
        display(pd.read_csv(fold_summary))
    else:
        print("Missing fold_summary.csv:", fold_summary)

    if sample_records.exists():
        print("Sample records:", sample_records)
    else:
        print("Missing sample_records.csv:", sample_records)


## 运行完整 governance 实验


In [4]:
import subprocess

cmd = [
    sys.executable,
    "scripts/run_governance_experiments.py",
    "--config",
    CONFIG_PATH,
]

print("运行命令：", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)

print("标准输出：")
print(result.stdout)

print("错误输出：")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"实验运行失败，返回码：{result.returncode}")

show_governance_summary("full")


运行命令： d:\Anaconda3\python.exe scripts/run_governance_experiments.py --config configs/telco_churn.yaml
标准输出：
【INFO】【2026-05-18 19:49:27】【数据加载】文本表格 E:\yan\组\三支决策\机器学习\BT_TWD_v2\data\Telco-Customer-Churn\Telco-Customer-Churn.csv 已读取，样本数=7043，列数=21
【INFO】【2026-05-18 19:49:27】【数据加载】5174 条标签无法映射，未指定负类且未开启 dropna_target，已按 0 处理
【INFO】【2026-05-18 19:49:27】【数据加载】标签列 Churn 已处理完成：dropna_target=False, 丢弃样本=0, 最终样本数=7043, 正类比例=26.54%
【INFO】【2026-05-18 19:49:27】【数据集信息】名称=telco_churn，样本数=7043，目标列=Churn，正类比例=26.54%
【INFO】【2026-05-18 19:49:27】【预处理】缺失值填充策略=most_frequent
【INFO】【2026-05-18 19:49:27】【预处理】连续特征=3个，类别特征=16个
【INFO】【2026-05-18 19:49:27】【预处理】编码后维度=30
【INFO】【2026-05-18 19:49:27】【governance】telco_churn 第 1/5 折
【INFO】【2026-05-18 19:49:28】【桶树】已为样本生成桶ID，共 35 个组合
【INFO】【2026-05-18 19:49:28】【桶树】已为样本生成桶ID，共 33 个组合
【INFO】【2026-05-18 19:49:34】【governance】weak bucket resolver：strong=13，weak=38
【INFO】【2026-05-18 19:49:34】【桶树】已为样本生成桶ID，共 33 个组合
【INFO】【2026-05-18 19:49:37】【governance】telco_churn 第 2/5 折
【INFO

,dataset_name,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,closure_rate,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,telco_churn,0.4562,0.753344,0.607988,0.711912,0.437176,0.749073,0.605928,0.719434,1.0,...,0.6,0.388889,0.611111,0.4,0.608696,0.217391,0.391304,0.211111,11.0,10.0


Reading: outputs\governance\full\telco_churn\fold_summary.csv


,dataset_name,fold_id,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,telco_churn,1,0.437899,0.763410,0.619746,0.723918,0.431867,0.759153,0.620332,0.740241,...,1.0,0.000000,1.000000,0.0,1.000000,0.400000,0.600000,1.000000,3.0,0.0
1,telco_churn,2,0.462740,0.745424,0.597744,0.696238,0.425124,0.751086,0.608871,0.724627,...,0.0,0.250000,0.750000,1.0,0.500000,0.333333,0.166667,-0.250000,3.0,10.0
2,telco_churn,3,0.451384,0.756389,0.611650,0.716111,0.425834,0.750154,0.606541,0.718240,...,1.0,0.000000,0.000000,0.0,1.000000,1.000000,0.000000,1.000000,0.0,0.0
3,telco_churn,4,0.443892,0.758266,0.612560,0.715199,0.444957,0.745440,0.597723,0.698864,...,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0
4,telco_churn,5,0.485085,0.743230,0.598240,0.708097,0.458097,0.739532,0.596173,0.715199,...,0.0,0.545455,0.454545,0.0,0.454545,0.000000,0.454545,-0.545455,5.0,0.0


Sample records: outputs\governance\full\telco_churn\sample_records.csv


## 运行无 CP 消融


In [5]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-cp-validation",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无 CP 消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_cp")
else:
    print("RUN_ABLATION=False，跳过无 CP 消融。")


RUN_ABLATION=False，跳过无 CP 消融。


## 运行无渐进更新消融


In [6]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-progressive-update",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无渐进更新消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_progressive")
else:
    print("RUN_ABLATION=False，跳过无渐进更新消融。")


RUN_ABLATION=False，跳过无渐进更新消融。
